# LACRIMAE — F04 SIGNUM
> *"Le Sceau est apposé. La vidéo devient artefact livrable."*

**Mission** : Post-production FFmpeg — filtres visuels (grain, contrast, sepia) + finalisation `short_master.mp4`

**Prérequis** :
- `F04_SIGNUM/IN/short_final.mp4` — de F03 PICTOR
- `F04_SIGNUM/IN/timing.json` — de F01 CANTOR
- `F04_SIGNUM/IN/creative_config.json` — de F02 VISIO
- `LAC_CUSTOS.py` dans `DRIVE_LACRIMAE/`

---

## ÉTAPE 1 — Montage Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive monté.')

## ÉTAPE 2 — Vérification des inputs (LAC_CUSTOS check-in)

In [ ]:
from pathlib import Path
import shutil, subprocess

DRIVE_BASE   = Path('/content/drive/MyDrive/DRIVE_LACRIMAE')
F04_BASE     = DRIVE_BASE / 'F04_SIGNUM'
IN_DIR       = F04_BASE / 'IN'
OUT_DIR      = F04_BASE / 'OUT'
CODEBASE_DIR = F04_BASE / 'CODEBASE'
CUSTOS_PATH  = DRIVE_BASE / 'LAC_CUSTOS.py'

OUT_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy(CUSTOS_PATH, '/content/LAC_CUSTOS.py')
result = subprocess.run(
    ['python', '/content/LAC_CUSTOS.py', '--frigate', 'F04',
     '--mode', 'check-in', '--drive-base', str(DRIVE_BASE)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('[SIGNUM] ERREUR check-in — corriger avant de continuer.')
    print(result.stderr)

## ÉTAPE 3 — Installation FFmpeg

In [ ]:
!apt-get install -y ffmpeg -q
!ffmpeg -version | head -1
print('FFmpeg installé.')

## ÉTAPE 4 — Copie du script SIGNUM

In [ ]:
import shutil, urllib.request

# Toujours tirer la dernière version depuis GitHub
GITHUB_RAW = (
    'https://raw.githubusercontent.com/kioka8877-ux/LACRIMAE/main'
    '/F04_SIGNUM/CODEBASE/lac_f04_signum.py'
)
dest_drive = CODEBASE_DIR / 'lac_f04_signum.py'
dest_local = '/content/lac_f04_signum.py'

urllib.request.urlretrieve(GITHUB_RAW, str(dest_drive))
shutil.copy(str(dest_drive), dest_local)
print('[SIGNUM] lac_f04_signum.py mis a jour depuis GitHub.')
print(f'  Drive : {dest_drive}')
print(f'  Local : {dest_local}')


## ÉTAPE 5 — Chargement creative_config

In [ ]:
import sys

# Vider le cache module pour forcer le rechargement
if 'lac_f04_signum' in sys.modules:
    del sys.modules['lac_f04_signum']

sys.path.insert(0, '/content')
from lac_f04_signum import load_creative_config, build_vf_filter, print_filter_summary

CREATIVE_CONFIG = str(IN_DIR / 'creative_config.json')

cfg = load_creative_config(CREATIVE_CONFIG)
vf  = build_vf_filter(cfg)
print_filter_summary(cfg, vf)


## ÉTAPE 6 — Preview frame (avant / après filtres)

Extrait la frame centrale du MP4 et l'affiche avec/sans filtres.

In [ ]:
import base64
from IPython.display import HTML, display
from lac_f04_signum import preview_frame

INPUT_MP4 = str(IN_DIR / 'short_final.mp4')
PREVIEW_DIR = '/content/signum_preview'

before_path, after_path = preview_frame(INPUT_MP4, vf, PREVIEW_DIR)

def b64(p):
    with open(p, 'rb') as f:
        return base64.b64encode(f.read()).decode()

display(HTML(f'''
<div style="display:flex;gap:16px;font-family:monospace;background:#111;padding:12px;border-radius:8px">
  <div style="flex:1;text-align:center">
    <p style="color:#888;margin:0 0 6px">ORIGINAL</p>
    <img src="data:image/jpeg;base64,{b64(before_path)}"
         style="width:100%;max-width:360px;border-radius:4px">
  </div>
  <div style="flex:1;text-align:center">
    <p style="color:#e8d5a0;margin:0 0 6px">AVEC FILTRES</p>
    <img src="data:image/jpeg;base64,{b64(after_path)}"
         style="width:100%;max-width:360px;border-radius:4px">
  </div>
</div>
'''))
print('[SIGNUM] Preview generee — frame centrale.')

## ÉTAPE 7 — Ajustement des paramètres (optionnel)

Si la preview ne convient pas, décommenter les lignes `OVERRIDE_*`, ajuster les valeurs, relancer cette cellule puis **retourner à l'étape 6** pour vérifier avant le rendu final.

In [ ]:
import re, copy

# ── MODIFIER CES VALEURS DIRECTEMENT ────────────────────────────────────────
# grain   : 0.0 (aucun) → 1.0 (max)
# contrast: 1.0 (neutre) → 1.5 (fort)
# bright  : 1.0 (neutre) → 0.7 (sombre)
# sepia   : 0.0 (aucun) → 1.0 (max)
OVERRIDE_GRAIN      = 0.15
OVERRIDE_CONTRAST   = 1.10
OVERRIDE_BRIGHTNESS = 0.90
OVERRIDE_SEPIA      = 0.55
# ─────────────────────────────────────────────────────────────────────────────

cfg_override = copy.deepcopy(cfg)
cfg_override['grain_overlay_opacity'] = OVERRIDE_GRAIN

css = {
    'contrast':   OVERRIDE_CONTRAST,
    'brightness': OVERRIDE_BRIGHTNESS,
    'sepia':      OVERRIDE_SEPIA,
}
cfg_override['css_filters'] = (
    f"contrast({css['contrast']}) "
    f"brightness({css['brightness']}) "
    f"sepia({css['sepia']})"
)

vf = build_vf_filter(cfg_override)
cfg = cfg_override
print_filter_summary(cfg, vf)
print('[SIGNUM] Overrides actifs — relancer etape 6 pour preview, puis etape 8 pour rendu.')


## ÉTAPE 8 — Rendu final FFmpeg

In [ ]:
from lac_f04_signum import finalize
import re

# Extraire les overrides actifs depuis cfg
css = {k: float(v) for k, v in re.findall(r'(\w+)\(([\d.]+)\)',
                                            cfg.get('css_filters', ''))}

ok = finalize(
    input_mp4        = str(IN_DIR  / 'short_final.mp4'),
    timing_json      = str(IN_DIR  / 'timing.json'),
    output_mp4       = str(OUT_DIR / 'short_master.mp4'),
    creative_config  = CREATIVE_CONFIG,
    title            = 'LACRIMAE',
    comment          = "For the Angel's Tears shall become gold. — Ad Victoriam.",
    override_grain      = cfg.get('grain_overlay_opacity'),
    override_contrast   = css.get('contrast'),
    override_brightness = css.get('brightness'),
    override_sepia      = css.get('sepia'),
)

if not ok:
    print('[SIGNUM] ÉCHEC — vérifier les logs.')

## ÉTAPE 9 — Validation LAC_CUSTOS check-out

In [ ]:
!python /content/LAC_CUSTOS.py --frigate F04 --mode check-out --drive-base "{DRIVE_BASE}"

## ÉTAPE 10 — Téléchargement du livrable

In [ ]:
from google.colab import files
from pathlib import Path

master_path = OUT_DIR / 'short_master.mp4'
if master_path.exists():
    size_mb = master_path.stat().st_size / 1_000_000
    print(f'[SIGNUM] Livrable prêt : {master_path} ({size_mb:.1f} Mo)')
    files.download(str(master_path))
    print('[SIGNUM] Téléchargement lancé.')
else:
    print('[SIGNUM] Livrable introuvable — étapes précédentes ont-elles réussi ?')

## ÉTAPE 11 — Mission accomplie

```
✓ short_master.mp4 téléchargé

  Inscrire dans TRACKING/LACRIMAE_CAMPAIGN_LOG.md :
  F04 SIGNUM → SCELLÉE

  Inscrire dans TRACKING/LACRIMAE_TRANSFER_LOG.md :
  F04 → Magos | short_master.mp4 | date

  Objectif suivant : Fleet Seal Certificate + publication
```

> *LACRIMAE — Né des larmes de Sanguinius, forgé en or.*
> *Ad Victoriam.*